In [49]:
import sys
sys.path.insert(0, '../src')
from models import load_model
from utils import *
from tversky_utils import *

config = parse_config('../configs/tversky_proj.yaml')
# latent dim 6, fbank size 128

model = load_model(config, "../results/tversky_proj_gridrobot/tversky_proj_1784130650.pth")
data = load_data(config)
all_trajs = data["trajs"]
all_feats = data["features"]

loading data: gridrobot_1960


# prediction
* sample high feature value trajs and label them "hi", low feature value trajs and label them "lo"
* find one (hi,lo) pair and do maxmin, minmax queries
* take another (hi, lo pair) and for each traj: compute similarity to hi features, similarity to lo features (or salience?)
    * print results

In [50]:
import numpy as np
import itertools
import torch

In [51]:
all_feats.shape # laptop, table

(1960, 2)

In [52]:
# using laptop (feature idx 0) because it is more query-able than upright, 
# see experiments/002-tversky-query-eval/figs/tversky_proj_fbank_size.png
hi_indices = np.where(all_feats[:,0] == np.max(all_feats[:,0]))[0]
lo_indices = np.where(all_feats[:,0] == np.min(all_feats[:,0]))[0]
print(f"{len(hi_indices)} hi indices, {len(lo_indices)} lo indices")
hi_trajs = all_trajs[hi_indices]
lo_trajs = all_trajs[lo_indices]

56 hi indices, 448 lo indices


In [53]:
pairs = np.array(list(itertools.product(hi_indices, lo_indices)))    # all (max_traj, min_traj) pairs)
pairs.shape

(25088, 2)

In [ ]:
TOP_FEATURE_COUNT = 128 # this is the feature bank size, so query will retrieve all salient features
TOP_RESULT_COUNT = 5
feature_bank = model.encoder[0].feature_bank.weight.detach()  # (F, D)
trajs_t = torch.as_tensor(all_trajs, dtype=torch.float32)
centered_trajs = (trajs_t - trajs_t.mean(0)).detach()       # (N, D)
def run_query(a_idx, b_idx):
    """s(a) - s(b): retrieve instances salient for a's features but not b's."""
    return retrieve_semantic_expression(
        instance_vectors=centered_trajs,
        feature_bank=feature_bank,
        expression=f"s({a_idx})-s({b_idx})",
        top_feature_count=TOP_FEATURE_COUNT,
        top_result_count=TOP_RESULT_COUNT,
    )



In [55]:
pair = pairs[0]
m,n = pair
res_maxmin = run_query(m, n)   # s(max) - s(min)
res_minmax = run_query(n, m)   # s(min) - s(max)

In [56]:
res_maxmin

{'expression': 's(29)-s(1)',
 'query_item_ixes': [29, 1],
 'feature_count': 51,
 'top_instances': [{'item_ix': 128,
   'salience': 86.81338500976562,
   'measure': 86.81338500976562},
  {'item_ix': 29, 'salience': 83.70231628417969, 'measure': 83.70231628417969},
  {'item_ix': 878,
   'salience': 80.59123992919922,
   'measure': 80.59123992919922},
  {'item_ix': 293, 'salience': 80.199951171875, 'measure': 79.64033508300781},
  {'item_ix': 885,
   'salience': 78.05712127685547,
   'measure': 77.90460205078125}],
 'semantic_features': {0,
  1,
  4,
  10,
  11,
  15,
  17,
  19,
  20,
  22,
  23,
  25,
  26,
  27,
  28,
  32,
  41,
  44,
  45,
  46,
  50,
  51,
  58,
  61,
  62,
  64,
  67,
  68,
  69,
  70,
  73,
  75,
  76,
  79,
  81,
  86,
  87,
  88,
  89,
  93,
  98,
  99,
  101,
  107,
  108,
  110,
  115,
  118,
  121,
  125,
  126}}

In [57]:
res_minmax

{'expression': 's(1)-s(29)',
 'query_item_ixes': [1, 29],
 'feature_count': 19,
 'top_instances': [{'item_ix': 1408,
   'salience': 25.760372161865234,
   'measure': 25.760372161865234},
  {'item_ix': 822,
   'salience': 26.527915954589844,
   'measure': 25.4534854888916},
  {'item_ix': 1545,
   'salience': 25.22469711303711,
   'measure': 25.22469711303711},
  {'item_ix': 1597,
   'salience': 26.063980102539062,
   'measure': 24.917810440063477},
  {'item_ix': 1300,
   'salience': 24.689023971557617,
   'measure': 24.689023971557617}],
 'semantic_features': {2,
  9,
  13,
  16,
  30,
  53,
  54,
  59,
  71,
  72,
  78,
  82,
  85,
  91,
  97,
  103,
  104,
  114,
  117}}

In [58]:
res_minmax['semantic_features']

{2, 9, 13, 16, 30, 53, 54, 59, 71, 72, 78, 82, 85, 91, 97, 103, 104, 114, 117}

In [59]:
test_pair = pairs[-1]
test_hi, test_lo = test_pair

In [60]:
import torch.nn.functional as F

def selective_salience(test_traj_idx, query_res):
    # get resulting features of this query
    semantic_features = query_res['semantic_features']
    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features))
    )
    centered_test_traj = centered_trajs[test_traj_idx]
    dot = centered_test_traj @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=1)                # (N,)
    # p_measures  = dot.sum(dim=1)                        # (N,)
    return p_saliences

In [61]:
maxmin_salience = selective_salience(torch.tensor(test_pair), res_maxmin)
minmax_salience = selective_salience(torch.tensor(test_pair), res_minmax)

print("                | hi traj  |  lo traj")
print(f"maxmin salience | {maxmin_salience[0]} | {maxmin_salience[1]}")
print(f"minmax salience | {minmax_salience[0]} | {minmax_salience[1]}")

                | hi traj  |  lo traj
maxmin salience | 14.339900970458984 | 18.17135238647461
minmax salience | 4.678924560546875 | 21.282440185546875


^ this is promising - high trajectory has greater salience with `res_maxmin` (which was hi - lo) and low trajectory has greater salience with `res_minmax` (which was lo - hi)!!

prediction:
take selective salience with res_maxmin

In [62]:
def tversky_predict(test_traj):
    hi_salience = selective_salience(torch.tensor([test_traj]), res_maxmin)
    lo_salience = selective_salience(torch.tensor([test_traj]), res_minmax)

    # TODO uncertainty measurement? softmax or sigmoid or something else...

    if hi_salience > lo_salience:
        return "hi"
    return "lo"


In [63]:
tversky_predict(test_lo)

'lo'

In [64]:

tversky_predict(test_hi)

'hi'